In [ ]:
%sql
SELECT * FROM `databricks_virtue_foundation_dataset_dais_2026`.`virtue_foundation_dataset`.`facilities`;

In [ ]:
%sql
SHOW TABLES IN databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset;

In [ ]:
%sql
DESCRIBE databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities;

In [ ]:
CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"
fac = spark.table(f"{CATALOG}.{SCHEMA}.facilities")

cols = ["name","address_city","address_stateOrRegion","address_zipOrPostcode",
        "latitude","longitude","coordinates","numberDoctors","capacity",
        "cluster_id","organization_type","facilityTypeId"]
display(fac.select(cols).limit(15))

In [ ]:
row = fac.filter("equipment is not null and procedure is not null").limit(3).collect()
for r in row:
    print("NAME:", r["name"])
    for c in ["specialties","procedure","equipment","capability"]:
        print(f"--- {c} ---")
        print(str(r[c])[:600])
    print("=" * 60)

In [ ]:
from pyspark.sql import functions as F
total = fac.count()
miss = fac.select([(1 - F.count(c)/total).alias(c) for c in fac.columns])
display(miss)

print("unique_id distinct:", fac.select("unique_id").distinct().count())
print("cluster_id distinct:", fac.select("cluster_id").distinct().count())
display(fac.groupBy("cluster_id").count().filter("count > 1").orderBy(F.desc("count")).limit(20))
display(fac.groupBy("name").count().filter("count > 1").orderBy(F.desc("count")).limit(20))

In [ ]:
%sql
DESCRIBE databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.india_post_pincode_directory;

In [ ]:
%sql
SHOW TABLES IN databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset LIKE 'nfhs*';

In [ ]:
from pyspark.sql import functions as F
CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"
fac = spark.table(f"{CATALOG}.{SCHEMA}.facilities")
N = fac.count()

EMPTY = ["", "null", "none", "nan", "[]", "n/a", "na"]
def is_empty(c):
    return F.col(c).isNull() | F.lower(F.trim(F.col(c).cast("string"))).isin(EMPTY)

key_cols = ["name","address_city","address_stateOrRegion","address_zipOrPostcode","latitude","longitude",
            "numberDoctors","capacity","yearEstablished","specialties","procedure","equipment",
            "capability","description","phone_numbers","officialPhone","websites","source_urls"]
r = fac.select([(F.sum(is_empty(c).cast("int"))/N).alias(c) for c in key_cols]).collect()[0].asDict()
print("总行数:", N)
for k, v in sorted(r.items(), key=lambda x: -x[1]):
    print(f"{k:28s}{v:7.1%}")

In [ ]:
pin = spark.table(f"{CATALOG}.{SCHEMA}.india_post_pincode_directory")
pin_c = (pin
    .withColumn("lat", F.expr("try_cast(latitude as double)"))
    .withColumn("lon", F.expr("try_cast(longitude as double)"))
    .filter(F.col("lat").between(6, 38) & F.col("lon").between(68, 98))
    .groupBy("pincode")
    .agg(F.percentile_approx("lat", 0.5).alias("pin_lat"),
         F.percentile_approx("lon", 0.5).alias("pin_lon"),
         F.expr("mode(statename)").alias("pin_state"),
         F.expr("mode(district)").alias("pin_district"),
         F.count("*").alias("n_offices")))
print("有可用坐标的邮编数:", pin_c.count(), "/ 邮编表总邮编数:", pin.select("pincode").distinct().count())
display(pin_c.limit(5))

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = (F.col(c) if isinstance(c, str) else c
                              for c in (lat1, lon1, lat2, lon2))
    dlat = F.radians(lat2 - lat1); dlon = F.radians(lon2 - lon1)
    a = F.sin(dlat/2)**2 + F.cos(F.radians(lat1))*F.cos(F.radians(lat2))*F.sin(dlon/2)**2
    return 2*6371.0*F.asin(F.sqrt(a))

f = (fac
    .withColumn("pin_str", F.regexp_extract("address_zipOrPostcode", r"(\d{6})", 1))
    .withColumn("pin", F.expr("try_cast(pin_str as bigint)"))
    .join(pin_c, F.col("pin") == pin_c["pincode"], "left"))

f = (f
    .withColumn("coord_in_india", F.col("latitude").between(6, 38) & F.col("longitude").between(68, 98))
    .withColumn("km_to_pin", haversine_km("latitude","longitude","pin_lat","pin_lon"))
    .withColumn("loc_flag",
        F.when(F.col("latitude").isNull() | F.col("longitude").isNull(), "no_coords")
         .when(~F.col("coord_in_india"), "outside_india")
         .when(F.col("pin_lat").isNull(), "ok_but_pin_unmatched")
         .when(F.col("km_to_pin") > 50, "far_from_pincode")
         .otherwise("ok"))
    # 坐标不可用时，退回到邮编质心，并记录来源
    .withColumn("coord_source",
        F.when(F.col("loc_flag").isin("ok","ok_but_pin_unmatched","far_from_pincode"), "facility")
         .when(F.col("pin_lat").isNotNull(), "pincode_centroid")
         .otherwise("none"))
    .withColumn("lat_final", F.when(F.col("coord_source")=="facility", F.col("latitude")).otherwise(F.col("pin_lat")))
    .withColumn("lon_final", F.when(F.col("coord_source")=="facility", F.col("longitude")).otherwise(F.col("pin_lon"))))

display(f.groupBy("loc_flag","coord_source").count().orderBy(F.desc("count")))
display(f.filter("loc_flag in ('outside_india','far_from_pincode')")
         .select("name","address_city","address_zipOrPostcode","pin_district","latitude","longitude","km_to_pin","loc_flag")
         .limit(30))

# Care Finder cleaning pipeline (steps 0–7)

Takes the shared facilities table (10,088 rows) to `facilities_final` (9,989 facilities).
Principles: **flag, do not delete** (only broken rows and exact duplicates are removed);
**impute location only** (missing positions fall back to the PIN centroid, labelled);
**empty means no evidence** (service fields are never filled in).

| Step | What it does | Result (verified) |
| --- | --- | --- |
| 0 | Convert literal `"null"`, `"none"`, `"n/a"`, `""`, `"[]"` to NULL in every text column and array element | — |
| 1 | Keep only `address_country = 'India'` | 10,088 → 10,000 |
| 2 | One row per `unique_id` (most complete row wins; tie-break on `source_content_id`, then full-row hash so runs are deterministic) | 10,000 → 9,989 |
| 3 | Parse capability / procedure / equipment / source_urls / specialties from JSON strings into clean arrays (drop blanks & placeholders, dedupe within field; specialties lowercased and stripped to `a-z0-9`); try_cast doctors / beds / year to int; `cluster_size = 1` when `cluster_id` is NULL | 9,989 |
| 4 | Flags: `has_doctors`, `has_capacity`, `has_year`, `pin_valid` (strict 6-digit match), implausible: year outside 1800–2026, doctors > 2,000, beds > 5,000 | 7 / 7 / 1 flagged; 227 missing-or-malformed PINs |
| 5 | `pin_lookup`: one row per PIN, median coords after India box filter (lat 6–38, lon 68–98), district count, prefix centroid (first 3 digits) | 19,586 PINs; 36 without coords; 1,478 multi-district |
| 6 | `facilities_geo`: coords usable only if inside the India box; gap < 50 km to PIN or to prefix region = consistent | 8,238 original / 429 unchecked / 1,286 conflict / 35 pin_approx / 1 unlocatable |
| 7 | `facilities_final`: conflicts resolved by median position of trusted (original) facilities with the same city+state (≥3); winner within 50 km | 504 pin-confirmed / 149 coords-confirmed / 633 unresolved |

Output tables in `workspace.care_finder`: `pin_lookup`, `facilities_clean`, `facilities_geo`, `facilities_final` (downstream queries use `facilities_final.lat_final` / `lon_final` / `location_quality`).
All thresholds (50 km, the India box, 1800, 2,000, 5,000, ≥3 facilities per vote) are judgment calls and provisional.

In [ ]:
from pyspark.sql import functions as F, types as T, Window

SRC = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset"
EMPTY_TOKENS = ["", "null", "none", "nan", "n/a", "na"]

fac_raw = spark.table(f"{SRC}.facilities")
ALL_COLS = fac_raw.columns

# ---- Step 0: placeholders -> NULL in every text column ----
def norm_text(colname):
    s = F.trim(F.col(colname).cast("string"))
    return F.when(s.isNull() | F.lower(s).isin(EMPTY_TOKENS), F.lit(None).cast("string")).otherwise(s)

step0 = fac_raw
for c, t in fac_raw.dtypes:
    if t == "string":
        step0 = step0.withColumn(c, norm_text(c))

# ---- Step 1: India rows only ----
step1 = step0.filter(F.col("address_country") == "India")

# ---- Step 2: one row per unique_id, deterministic ----
completeness = sum(F.col(c).isNotNull().cast("int") for c in
    ["latitude","longitude","numberDoctors","capacity","description",
     "capability","procedure","equipment","specialties","phone_numbers"])
row_hash = F.sha2(F.to_json(F.struct(*[F.col(c) for c in ALL_COLS])), 256)
w = Window.partitionBy("unique_id").orderBy(F.desc(completeness),
                                            F.col("source_content_id").desc(),
                                            row_hash)
step2 = step1.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

# ---- Step 3: parse arrays, typed numbers ----
def parse_array(colname, element_transform=None):
    s = F.col(colname)
    s = F.when(s.isNull() | F.lower(F.trim(s)).isin(EMPTY_TOKENS + ["[]"]), F.lit(None).cast("string")).otherwise(s)
    arr = F.from_json(s, T.ArrayType(T.StringType()))
    arr = F.filter(arr, lambda x: x.isNotNull() & (F.length(F.trim(x)) > 0)
                   & ~F.lower(F.trim(x)).isin(EMPTY_TOKENS))
    if element_transform is not None:
        arr = F.filter(F.transform(arr, element_transform), lambda x: F.length(x) > 0)
    else:
        arr = F.transform(arr, lambda x: F.trim(x))
    return F.array_distinct(arr)

# ---- Step 4: presence + plausibility flags ----
clean = (step2
    .withColumn("capability_list",  parse_array("capability"))
    .withColumn("procedure_list",   parse_array("procedure"))
    .withColumn("equipment_list",  parse_array("equipment"))
    .withColumn("source_url_list",  parse_array("source_urls"))
    .withColumn("specialties_tags", parse_array("specialties",
        lambda x: F.regexp_replace(F.lower(F.trim(x)), "[^a-z0-9]", "")))
    .withColumn("doctors", F.expr("try_cast(numberDoctors as int)"))
    .withColumn("beds",     F.expr("try_cast(capacity as int)"))
    .withColumn("year",     F.expr("try_cast(yearEstablished as int)"))
    .withColumn("cluster_size", F.when(F.col("cluster_id").isNull(), F.lit(1))
        .otherwise(F.count(F.lit(1)).over(Window.partitionBy("cluster_id"))))
    .withColumn("pin_str", F.regexp_extract("address_zipOrPostcode", r"^(\d{6})$", 1))
    .withColumn("pin_valid", F.col("pin_str").isNotNull() & (F.col("pin_str") != ""))
    .withColumn("has_doctors", F.col("doctors").isNotNull())
    .withColumn("has_capacity", F.col("beds").isNotNull())
    .withColumn("has_year",     F.col("year").isNotNull())
    .withColumn("year_implausible",    F.col("has_year") & ~F.col("year").between(1800, 2026))
    .withColumn("doctors_implausible", F.col("has_doctors") & (F.col("doctors") > 2000))
    .withColumn("beds_implausible",     F.col("has_capacity") & (F.col("beds") > 5000)))

print("facilities_clean:", clean.count(),
      "| pin invalid:", clean.filter("~pin_valid").count(),
      "| bad year:", clean.filter("year_implausible").count(),
      "| bad docs:", clean.filter("doctors_implausible").count(),
      "| bad beds:", clean.filter("beds_implausible").count())

In [ ]:
# ---- Step 5: pin_lookup + prefix-region centroids ----
pin = spark.table(f"{SRC}.india_post_pincode_directory")
pin_box = (pin
    .withColumn("lat", F.expr("try_cast(latitude as double)"))
    .withColumn("lon", F.expr("try_cast(longitude as double)"))
    .filter(F.col("lat").between(6, 38) & F.col("lon").between(68, 98)))

pin_lookup = (pin.groupBy("pincode")
    .agg(F.expr("count(distinct district)").alias("district_count"),
         F.expr("mode(statename)").alias("pin_state"),
         F.expr("mode(district)").alias("pin_district"),
         F.count("*").alias("n_offices"))
    .join(pin_box.groupBy("pincode")
          .agg(F.percentile_approx("lat", 0.5).alias("pin_lat"),
               F.percentile_approx("lon", 0.5).alias("pin_lon")),
          "pincode", "left")
    .withColumn("region_prefix", (F.col("pincode")/1000).cast("int")))

region_lookup = (pin_lookup.filter("pin_lat is not null")
    .groupBy("region_prefix")
    .agg(F.percentile_approx("pin_lat", 0.5).alias("region_lat"),
         F.percentile_approx("pin_lon", 0.5).alias("region_lon")))

print("pin_lookup PINs:", pin_lookup.count(),
      "| no coords:", pin_lookup.filter("pin_lat is null").count(),
      "| multi-district:", pin_lookup.filter("district_count > 1").count())

In [ ]:
# ---- Step 6: facilities_geo ----
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = (F.col(c) if isinstance(c, str) else c
                              for c in (lat1, lon1, lat2, lon2))
    dlat = F.radians(lat2 - lat1); dlon = F.radians(lon2 - lon1)
    a = F.sin(dlat/2)**2 + F.cos(F.radians(lat1))*F.cos(F.radians(lat2))*F.sin(dlon/2)**2
    return 2*6371.0*F.asin(F.sqrt(a))

geo = (clean
    .withColumn("pin", F.expr("try_cast(pin_str as bigint)"))
    .join(pin_lookup.select("pincode","pin_lat","pin_lon","pin_state","pin_district",
                            "district_count","region_prefix").withColumnRenamed("pincode","pin"),
          "pin", "left")
    .join(region_lookup, "region_prefix", "left")
    .withColumn("coords_usable",
        F.col("latitude").isNotNull() & F.col("longitude").isNotNull()
        & F.col("latitude").between(6, 38) & F.col("longitude").between(68, 98))
    .withColumn("km_to_pin", F.when(F.col("coords_usable") & F.col("pin_lat").isNotNull(),
                                    haversine_km("latitude","longitude","pin_lat","pin_lon")))
    .withColumn("km_to_region", F.when(F.col("coords_usable") & F.col("region_lat").isNotNull(),
                                       haversine_km("latitude","longitude","region_lat","region_lon")))
    .withColumn("pin_checkable", F.col("km_to_pin").isNotNull() | F.col("km_to_region").isNotNull())
    .withColumn("geo_status",
        F.when(~F.col("coords_usable"),
               F.when(F.col("pin_lat").isNotNull(), "pin_approx").otherwise("unlocatable"))
         .when(~F.col("pin_checkable"), "original_unchecked")
         .when((F.col("km_to_pin") < 50) | (F.col("km_to_region") < 50), "original")
         .otherwise("conflict")))

display(geo.groupBy("geo_status").count().orderBy(F.desc("count")))

In [ ]:
# ---- Step 7: resolve conflicts by same-city vote, then final position ----
trusted = (geo.filter(F.col("geo_status") == "original")
    .groupBy("address_city", "address_stateOrRegion")
    .agg(F.percentile_approx("latitude", 0.5).alias("vote_lat"),
         F.percentile_approx("longitude", 0.5).alias("vote_lon"),
         F.count("*").alias("vote_n"))
    .filter(F.col("vote_n") >= 3))

resolved = (geo.filter(F.col("geo_status") == "conflict")
    .join(trusted, ["address_city", "address_stateOrRegion"], "left")
    .withColumn("km_coords_to_vote", haversine_km("latitude","longitude","vote_lat","vote_lon"))
    .withColumn("km_pin_to_vote",    haversine_km("pin_lat","pin_lon","vote_lat","vote_lon"))
    .withColumn("resolution",
        F.when(F.col("vote_lat").isNull(), "unresolved")
         .when((F.col("km_coords_to_vote") <= 50) & (F.col("km_pin_to_vote") > 50), "coords_confirmed")
         .when((F.col("km_pin_to_vote") <= 50) & (F.col("km_coords_to_vote") > 50), "pin_confirmed")
         .otherwise("unresolved")))

final = (geo
    .join(resolved.select("unique_id","vote_lat","vote_lon","vote_n",
                          "km_coords_to_vote","km_pin_to_vote","resolution"),
          "unique_id", "left")
    .withColumn("location_quality",
        F.when(F.col("geo_status") != "conflict", F.col("geo_status"))
         .when(F.col("resolution") == "pin_confirmed",    "conflict_pin_confirmed")
         .when(F.col("resolution") == "coords_confirmed", "conflict_coords_confirmed")
         .otherwise("conflict_unresolved"))
    .withColumn("use_coords",
        F.col("location_quality").isin("original","original_unchecked","conflict_coords_confirmed"))
    .withColumn("lat_final", F.when(F.col("location_quality") == "unlocatable", F.lit(None).cast("double"))
         .when(F.col("use_coords"), F.col("latitude")).otherwise(F.col("pin_lat")))
    .withColumn("lon_final", F.when(F.col("location_quality") == "unlocatable", F.lit(None).cast("double"))
         .when(F.col("use_coords"), F.col("longitude")).otherwise(F.col("pin_lon")))
    .drop("use_coords"))

display(final.groupBy("location_quality").count().orderBy(F.desc("count")))

In [ ]:
# ---- Write all four tables ----
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.care_finder")

pin_lookup.write.mode("overwrite").saveAsTable("workspace.care_finder.pin_lookup")
clean.write.mode("overwrite").saveAsTable("workspace.care_finder.facilities_clean")
geo.write.mode("overwrite").saveAsTable("workspace.care_finder.facilities_geo")
final.write.mode("overwrite").saveAsTable("workspace.care_finder.facilities_final")

# ---- Independent verification ----
for t in ["pin_lookup", "facilities_clean", "facilities_geo", "facilities_final"]:
    print(t, spark.table(f"workspace.care_finder.{t}").count())

spark.sql("""
SELECT location_quality, COUNT(*) n
FROM workspace.care_finder.facilities_final
GROUP BY location_quality ORDER BY n DESC
""").show()

# Steps 8–11: contamination flags, citations, maternal evidence, near-duplicates

Runs **after** the steps 0–7 cells. Principle stays **flag, do not delete**: contaminated records and
near-duplicates are flagged and cited, never removed or auto-split.

| Step | What it does | Result (verified) |
| --- | --- | --- |
| 8 | `nearby_sentence_count` (proximity phrases), `distinct_city_count` ("Located in X" in capability), `n_source_urls`, and near-duplicate flags: same normalized name + city → `near_dup_flag`, `near_dup_n`, `near_dup_conflict_fields` | 336 proximity / 1,126 multi-city / 715 with ≥50 URLs; 10 rows in near-dup groups |
| 9 | `service_taxonomy` (10 maternal/newborn services with text + strong-tag regex); other-facility citation detection: extract capitalized facility-name candidates from every sentence of capability/procedure/equipment/description and match against the 9,989-name list → `facility_other_facility_mentions` | 448,125 sentences scanned; 4,676 cite another facility; 1,255 facilities affected |
| 10 | `facility_service_evidence`: every claim posexploded with field + position + quote, tiered (specialties=strong, procedure=medium, capability/equipment/description=weak); rejected if negated (no/not/none/planned/referral) or citing another facility; `facility_service_summary` per facility + service | 49,356 evidence rows; 48,220 accepted; 48 negated; 88 other-facility |
| 11 | Final flags on `facilities_final`: `other_facility_mention_count`, `contamination_flag`, `maternal_services`, `maternal_best_tier` | 2,317 contamination-flagged; 5,822 maternal facilities (strong 4,821 / medium 288 / weak 713) |

The Raj Eye Hospital case: its C-section claim now carries citations (a `gynecologyandobstetrics` tag and
"Performs emergency cesarean sections"), and the 88 evidence sentences that describe a different facility
are rejected with the other facility's name recorded. Near-duplicate **merging** is deliberately not done —
the flags are the working set for a reviewed merge. Schema-changing overwrites of `facilities_final` need
`.option("overwriteSchema", "true")`.

In [ ]:
# ---- Step 8: contamination + near-duplicate flags (reads facilities_final, rewrites it) ----
from pyspark.sql import functions as F

ff = spark.table("workspace.care_finder.facilities_final")

PROX = r"(?i)\b(nearby|opposite|adjacent to|next to|across from|behind|near the|in the vicinity)\b"
arr3 = F.concat(F.coalesce(F.col("capability_list"), F.array()),
               F.coalesce(F.col("procedure_list"),  F.array()),
               F.coalesce(F.col("equipment_list"),   F.array()))

ff2 = (ff
    .withColumn("nearby_sentence_count", F.size(F.filter(arr3, lambda x: x.rlike(PROX))))
    .withColumn("distinct_city_count", F.size(F.array_distinct(F.filter(
        F.transform(F.coalesce(F.col("capability_list"), F.array()),
                    lambda x: F.regexp_extract(x, r"(?i)^Located in (.+)$", 1)),
        lambda x: F.length(x) > 0))))
    .withColumn("n_source_urls", F.size(F.coalesce(F.col("source_url_list"), F.array()))))

# near-duplicates: normalized name + city (flag only, no merge)
def norm(c):
    return F.regexp_replace(F.lower(F.trim(c)), r"\s+", " ")

CONFLICT_FIELDS = ["address_zipOrPostcode","officialPhone","phone_numbers","websites",
                   "address_line1","numberDoctors","capacity","yearEstablished"]
ff2 = (ff2
    .withColumn("nd_name", norm("name"))
    .withColumn("nd_city", F.coalesce(norm("address_city"), F.lit(""))))
nd_stats = (ff2.groupBy("nd_name", "nd_city")
    .agg(F.count("*").alias("near_dup_n"),
         F.array(*[F.when(F.size(F.collect_set(F.when(F.col(c).isNotNull(), F.col(c)))) > 1,
                          F.lit(c)) for c in CONFLICT_FIELDS]).alias("conflict_arr"))
    .withColumn("near_dup_conflict_fields", F.filter("conflict_arr", lambda x: x.isNotNull()))
    .withColumn("near_dup_conflicts", F.size("near_dup_conflict_fields"))
    .drop("conflict_arr"))
ff2 = (ff2.join(nd_stats, ["nd_name","nd_city"], "left")
    .withColumn("near_dup_flag", F.col("near_dup_n") > 1)
    .drop("nd_name","nd_city"))

ff2.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.care_finder.facilities_final")

v = spark.table("workspace.care_finder.facilities_final")
print("rows:", v.count(),
      "| near-dup flagged:", v.filter("near_dup_flag").count(),
      "| proximity:", v.filter("nearby_sentence_count > 0").count(),
      "| >=3 cities:", v.filter("distinct_city_count >= 3").count())

In [ ]:
# ---- Step 9: service taxonomy + other-facility citation detection ----
from functools import reduce

ff = spark.table("workspace.care_finder.facilities_final")

TAXONOMY = [
 ("antenatal_care",       r"(?i)\banc\b|antenatal|prenatal|pregnan", "obstetric|gynec|midwi"),
 ("institutional_delivery", r"(?i)institutional deliver|labour room|labor room|\bdeliver", "obstetric|gynec"),
 ("normal_delivery",      r"(?i)normal deliver|vaginal deliver", "obstetric|gynec"),
 ("c_section",            r"(?i)c[- ]?section|cesar|caesar", "obstetric|gynec"),
 ("emergency_obstetric",  r"(?i)emergency obstetric|obstetric emergen|emoc", "obstetric|gynec"),
 ("newborn_care",         r"(?i)newborn|neonat|\bnicu\b|\bsncu\b|special newborn", "neonat"),
 ("postnatal_care",       r"(?i)postnatal|\bpnc\b|after delivery|post[- ]delivery", "obstetric|gynec"),
 ("family_planning",      r"(?i)family planning|contracept|sterili[sz]ation|tubectomy|vasectomy|\biud\b|intrauterine|oral pill", None),
 ("immunization",         r"(?i)immuni[sz]|vaccin|\bbcg\b|\bpolio\b|measles|pentavalent", None),
 ("breastfeeding_support", r"(?i)breastfeed|lactat|colostrum", None),
]
tax = spark.createDataFrame(TAXONOMY, "service string, text_regex string, strong_tag_regex string")
tax.write.mode("overwrite").saveAsTable("workspace.care_finder.service_taxonomy")

NAME_RE = (r"((?:[A-Z][\w&'.-]*\s+){0,4}[A-Z][\w&'.-]*\s+"
           r"(?:Hospital|Hospitals|Clinic|Nursing Home|Institute|Medical College|"
           r"College|Centre|Center|Maternity Home|Polyclinic|Sanatorium|Trust))")

# all sentences: three arrays + description sentences (literal field, no withColumn in the loop)
parts = [ff.select("unique_id", "name", F.lit(fld).alias("field"),
                   F.expr(f"posexplode({fld})")).withColumnRenamed("col", "quote")
         for fld in ["capability_list", "procedure_list", "equipment_list"]]
d = (ff.filter("description is not null")
     .select("unique_id", "name", F.split("description", r"(?<=[.!?])\s+").alias("s")))
parts.append(d.select("unique_id", "name", F.lit("description").alias("field"),
                      F.expr("posexplode(s)")).withColumnRenamed("col", "quote"))
all_items = (reduce(lambda a, b: a.unionByName(b), parts)
    .select("unique_id", "name", "field", "pos", "quote")
    .filter(F.length(F.trim("quote")) > 5))

names = (ff.select(norm("name").alias("facility_name")).distinct()
    .filter(F.length("facility_name") >= 8))

hits = (all_items
    .select("unique_id", "name", "field", "pos", "quote",
            F.explode(F.regexp_extract_all(F.col("quote"), F.lit(NAME_RE))).alias("cand"))
    .withColumn("cand_norm", norm(F.col("cand")))
    .join(names, F.col("cand_norm") == F.col("facility_name"))
    .filter(F.col("cand_norm") != norm(F.col("name")))
    .groupBy("unique_id", "field", "pos", "quote")
    .agg(F.min("cand_norm").alias("other_facility_name")))

print("sentences scanned:", all_items.count(),
      "| citing another facility:", hits.count(),
      "| facilities affected:", hits.select("unique_id").distinct().count())
hits.write.mode("overwrite").saveAsTable(
    "workspace.care_finder.facility_other_facility_mentions")

In [ ]:
# ---- Step 10: tiered evidence + summary (uses all_items, hits, tax from step 9) ----
NEG = r"(?i)\b(no|not|none|nor|neither|without|unavailable|planned|referral|referred to)\b"

ev_sents = (all_items.crossJoin(tax)
    .where(F.expr("quote rlike text_regex"))
    .withColumn("tier", F.when(F.col("field") == "procedure_list", "medium").otherwise("weak"))
    .withColumn("is_negated", F.col("quote").rlike(NEG))
    .join(hits, ["unique_id","field","pos","quote"], "left")
    .withColumn("is_other_facility", F.col("other_facility_name").isNotNull())
    .withColumn("accepted", ~F.col("is_negated") & ~F.col("is_other_facility")))

tag_ev = (ff.select("unique_id", "name", F.expr("posexplode(specialties_tags)"))
    .withColumnRenamed("col", "tag")
    .crossJoin(tax.filter("strong_tag_regex is not null"))
    .where(F.expr("tag rlike strong_tag_regex"))
    .select(F.col("unique_id"), F.col("service"), F.lit("strong").alias("tier"),
            F.lit("specialties").alias("field"), F.col("pos"), F.col("tag").alias("quote"),
            F.lit(False).alias("is_negated"),
            F.lit(None).cast("string").alias("other_facility_name"),
            F.lit(False).alias("is_other_facility"),
            F.lit(True).alias("accepted")))

evidence = (ev_sents.select("unique_id","service","tier","field","pos","quote",
                            "is_negated","is_other_facility","other_facility_name","accepted")
    .unionByName(tag_ev))
print("evidence rows:", evidence.count(),
      "| accepted:", evidence.filter("accepted").count(),
      "| rejected (negated):", evidence.filter("not accepted and is_negated").count(),
      "| rejected (other facility):", evidence.filter("not accepted and is_other_facility and not is_negated").count())
evidence.write.mode("overwrite").saveAsTable(
    "workspace.care_finder.facility_service_evidence")

rank = F.when(F.col("tier") == "strong", F.lit(1)).when(F.col("tier") == "medium", F.lit(2)).otherwise(F.lit(3))
summary = (evidence.withColumn("_rank", rank)
    .groupBy("unique_id", "service")
    .agg(F.sum(F.col("accepted").cast("int")).alias("n_citations"),
         F.min(F.when(F.col("accepted"), F.col("_rank"))).alias("_best"),
         F.sum(F.col("is_negated").cast("int")).alias("n_negated"),
         F.sum(F.col("is_other_facility").cast("int")).alias("n_other_facility"))
    .withColumn("best_tier", F.when(F.col("_best") == 1, "strong")
                .when(F.col("_best") == 2, "medium").when(F.col("_best") == 3, "weak"))
    .drop("_best"))
summary.write.mode("overwrite").saveAsTable(
    "workspace.care_finder.facility_service_summary")

# citation check: what backs Raj Eye Hospital's C-section claim?
raj = ff.filter(F.upper(F.col("name")).like("%RAJ EYE%")).select("unique_id", "name")
display(evidence.join(raj, "unique_id").filter("service = 'c_section'")
        .select("name","service","field","quote","other_facility_name","is_negated","accepted"))

In [ ]:
# ---- Step 11: contamination + maternal flags on facilities_final ----
ff = spark.table("workspace.care_finder.facilities_final")

mentions = (spark.table("workspace.care_finder.facility_other_facility_mentions")
    .groupBy("unique_id").agg(F.count("*").alias("other_facility_mention_count")))

t_rank = (F.when(F.col("best_tier") == "strong", F.lit(1))
           .when(F.col("best_tier") == "medium", F.lit(2))
           .when(F.col("best_tier") == "weak", F.lit(3)))
maternal = (spark.table("workspace.care_finder.facility_service_summary")
    .filter("n_citations > 0")
    .withColumn("_r", t_rank)
    .groupBy("unique_id")
    .agg(F.count_distinct("service").alias("_n_services"), F.min("_r").alias("_best_r"))
    .withColumn("maternal_services", F.col("_n_services"))
    .withColumn("maternal_best_tier",
        F.when(F.col("_best_r") == 1, "strong").when(F.col("_best_r") == 2, "medium")
         .when(F.col("_best_r") == 3, "weak"))
    .select("unique_id", "maternal_services", "maternal_best_tier"))

ff2 = (ff
    .join(mentions, "unique_id", "left")
    .join(maternal, "unique_id", "left")
    .withColumn("other_facility_mention_count", F.coalesce(F.col("other_facility_mention_count"), F.lit(0)))
    .withColumn("maternal_services", F.coalesce(F.col("maternal_services"), F.lit(0)))
    .withColumn("contamination_flag",
        (F.col("nearby_sentence_count") > 0) |
        (F.col("distinct_city_count") >= 3) |
        (F.col("other_facility_mention_count") > 0)))

ff2.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.care_finder.facilities_final")

v = spark.table("workspace.care_finder.facilities_final")
print("rows:", v.count(),
      "| contamination flagged:", v.filter("contamination_flag").count(),
      "| maternal facilities:", v.filter("maternal_services > 0").count())
display(v.groupBy("maternal_best_tier").agg(F.count("*").alias("facilities"),
                                           F.sum("maternal_services").alias("service_links")))

# Steps 12–15: model inputs — condition matrix, eligibility, district coverage

Runs after the steps 8–11 cells. Feeds the decision model's inputs (`M(j)`, `p_j`, J eligibility)
and Mode C's coverage view. Same principle: **flag, do not delete**.

| Step | What it does | Result (verified) |
| --- | --- | --- |
| 12 | Extend `service_taxonomy` with `blood_bank` and `emergency_24h`; rerun evidence extraction over all 448k sentences (other-facility citations reused from the table); overwrite evidence + summary | blood_bank 621 facilities (119 medium / 502 weak); emergency_24h 4,261 (592 medium / 3,669 weak); summary 36,929 rows |
| 13 | `facility_condition_matrix`: facility × condition (c_section, institutional_delivery, newborn_care, blood_bank, emergency_24h) with citations, best tier, and a tier-weighted condition strength (strong 1 / medium 0.5 / weak 0.25), halved when `contamination_flag` | 16,497 rows |
| 14 | `model_eligibility` on `facilities_final` from `location_quality` (eligible = original / original_unchecked / conflict-confirmed; warning = conflict_unresolved, pin_approx; excluded = unlocatable) plus the mother stand-in `mother_conditions_effective` / `mother_standin_p` (sum of condition weights over the 4 mother conditions, p = effective / 5) | 9,320 eligible / 668 warning / 1 excluded; C-section J 4,665 → 4,247; 6,121 facilities with mother evidence (avg p 0.204) |
| 15 | `district_service_coverage` (eligible facilities per district × condition, joined to NFHS-5 need measures on district name) and `district_care_gaps` (NFHS districts with no eligible facility for a condition) | 1,808 coverage rows over 468 districts; 1,299/1,808 matched NFHS; 2,231 gaps (blood_bank 554, newborn 481, emergency 406, c_section 400, institutional delivery 390) |

Notes:
- The NFHS-5 table lives in the source catalog
  `databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.nfhs_5_district_health_indicators`.
  The join is on lowercased district + state names; ~28% of coverage rows are unmatched (spelling variants),
  so treat the need columns as best-effort.
- `maternal_services` / `maternal_best_tier` (step 11) reflect the 10-service taxonomy snapshot; the model's
  mother input is `mother_conditions_effective` / `mother_standin_p` from step 14.
- The app should read `facility_condition_matrix` + `model_eligibility` + `contamination_flag` instead of
  keyword LIKE queries, so every condition claim is citation-backed and false positives are filtered.

In [ ]:
# ---- Step 12: extend taxonomy (blood_bank, emergency_24h) and rerun evidence ----
# Self-contained: re-derives the sentence universe; other-facility citations come from the table.
from functools import reduce

ff = spark.table("workspace.care_finder.facilities_final")

TAXONOMY = [
 ("antenatal_care",       r"(?i)\banc\b|antenatal|prenatal|pregnan", "obstetric|gynec|midwi"),
 ("institutional_delivery", r"(?i)institutional deliver|labour room|labor room|\bdeliver", "obstetric|gynec"),
 ("normal_delivery",      r"(?i)normal deliver|vaginal deliver", "obstetric|gynec"),
 ("c_section",            r"(?i)c[- ]?section|cesar|caesar", "obstetric|gynec"),
 ("emergency_obstetric",  r"(?i)emergency obstetric|obstetric emergen|emoc", "obstetric|gynec"),
 ("newborn_care",         r"(?i)newborn|neonat|\bnicu\b|\bsncu\b|special newborn", "neonat"),
 ("postnatal_care",       r"(?i)postnatal|\bpnc\b|after delivery|post[- ]delivery", "obstetric|gynec"),
 ("family_planning",      r"(?i)family planning|contracept|sterili[sz]ation|tubectomy|vasectomy|\biud\b|intrauterine|oral pill", None),
 ("immunization",         r"(?i)immuni[sz]|vaccin|\bbcg\b|\bpolio\b|measles|pentavalent", None),
 ("breastfeeding_support", r"(?i)breastfeed|lactat|colostrum", None),
 ("blood_bank",           r"(?i)blood bank|blood storage|safe blood|blood transfusion|blood donation|blood unit|storage of blood", None),
 ("emergency_24h",        r"(?i)24[- ]?hours?|24/7|round[- ]the[- ]clock|emergency (care|services|department|room)|casualty", None),
]
tax = spark.createDataFrame(TAXONOMY, "service string, text_regex string, strong_tag_regex string")
tax.write.mode("overwrite").saveAsTable("workspace.care_finder.service_taxonomy")

parts = [ff.select("unique_id", F.lit(fld).alias("field"),
                   F.expr(f"posexplode({fld})")).withColumnRenamed("col", "quote")
         for fld in ["capability_list", "procedure_list", "equipment_list"]]
d = (ff.filter("description is not null")
     .select("unique_id", F.split("description", r"(?<=[.!?])\s+").alias("s")))
parts.append(d.select("unique_id", F.lit("description").alias("field"),
                      F.expr("posexplode(s)")).withColumnRenamed("col", "quote"))
all_items = (reduce(lambda a, b: a.unionByName(b), parts)
    .select("unique_id", "field", "pos", "quote")
    .filter(F.length(F.trim("quote")) > 5))

hits = spark.table("workspace.care_finder.facility_other_facility_mentions")
NEG = r"(?i)\b(no|not|none|nor|neither|without|unavailable|planned|referral|referred to)\b"

ev_sents = (all_items.crossJoin(tax)
    .where(F.expr("quote rlike text_regex"))
    .withColumn("tier", F.when(F.col("field") == "procedure_list", "medium").otherwise("weak"))
    .withColumn("is_negated", F.col("quote").rlike(NEG))
    .join(hits, ["unique_id","field","pos","quote"], "left")
    .withColumn("is_other_facility", F.col("other_facility_name").isNotNull())
    .withColumn("accepted", ~F.col("is_negated") & ~F.col("is_other_facility")))

tag_ev = (ff.select("unique_id", F.expr("posexplode(specialties_tags)"))
    .withColumnRenamed("col", "tag")
    .crossJoin(tax.filter("strong_tag_regex is not null"))
    .where(F.expr("tag rlike strong_tag_regex"))
    .select(F.col("unique_id"), F.col("service"), F.lit("strong").alias("tier"),
            F.lit("specialties").alias("field"), F.col("pos"), F.col("tag").alias("quote"),
            F.lit(False).alias("is_negated"),
            F.lit(None).cast("string").alias("other_facility_name"),
            F.lit(False).alias("is_other_facility"),
            F.lit(True).alias("accepted")))

evidence = (ev_sents.select("unique_id","service","tier","field","pos","quote",
                            "is_negated","is_other_facility","other_facility_name","accepted")
    .unionByName(tag_ev))
evidence.write.mode("overwrite").saveAsTable("workspace.care_finder.facility_service_evidence")

rank = F.when(F.col("tier") == "strong", F.lit(1)).when(F.col("tier") == "medium", F.lit(2)).otherwise(F.lit(3))
summary = (evidence.withColumn("_rank", rank)
    .groupBy("unique_id", "service")
    .agg(F.sum(F.col("accepted").cast("int")).alias("n_citations"),
         F.min(F.when(F.col("accepted"), F.col("_rank"))).alias("_best"),
         F.sum(F.col("is_negated").cast("int")).alias("n_negated"),
         F.sum(F.col("is_other_facility").cast("int")).alias("n_other_facility"))
    .withColumn("best_tier", F.when(F.col("_best") == 1, "strong")
                .when(F.col("_best") == 2, "medium").when(F.col("_best") == 3, "weak"))
    .drop("_best"))
summary.write.mode("overwrite").saveAsTable("workspace.care_finder.facility_service_summary")

s = spark.table("workspace.care_finder.facility_service_summary").filter("n_citations > 0")
display(s.groupBy("service").agg(F.count("*").alias("facilities")).orderBy(F.desc("facilities")))
display(s.filter("service in ('blood_bank','emergency_24h')").groupBy("service", "best_tier").count())

In [ ]:
# ---- Step 13: facility_condition_matrix (the model's M(j)) ----
ff = spark.table("workspace.care_finder.facilities_final")
summ = spark.table("workspace.care_finder.facility_service_summary").filter("n_citations > 0")
CONDS = ["c_section", "institutional_delivery", "newborn_care", "blood_bank", "emergency_24h"]

tier_w = (F.when(F.col("best_tier") == "strong", F.lit(1.0))
          .when(F.col("best_tier") == "medium", F.lit(0.5)).otherwise(F.lit(0.25)))

matrix = (summ.filter(F.col("service").isin(CONDS))
    .join(ff.select("unique_id", "contamination_flag"), "unique_id")
    .withColumn("tier_weight", tier_w)
    .withColumn("condition_weight",
        F.round(F.col("tier_weight") * F.when(F.col("contamination_flag"), F.lit(0.5)).otherwise(F.lit(1.0)), 3))
    .select("unique_id", "service", "n_citations", "n_negated", "n_other_facility",
            "best_tier", "tier_weight", "contamination_flag", "condition_weight"))
matrix.write.mode("overwrite").saveAsTable("workspace.care_finder.facility_condition_matrix")
display(spark.table("workspace.care_finder.facility_condition_matrix")
        .groupBy("service").agg(F.count("*").alias("facilities")))

In [ ]:
# ---- Step 14: model eligibility + mother stand-in on facilities_final ----
ff = spark.table("workspace.care_finder.facilities_final")
m = spark.table("workspace.care_finder.facility_condition_matrix")

mother = (m.filter("service in ('c_section','blood_bank','emergency_24h','newborn_care')")
    .groupBy("unique_id")
    .agg(F.round(F.sum("condition_weight"), 3).alias("mother_conditions_effective")))
mother = mother.withColumn("mother_standin_p", F.round(F.col("mother_conditions_effective") / 5, 3))

me = (F.when(F.col("location_quality").isin("conflict_unresolved", "pin_approx"), "warning")
      .when(F.col("location_quality") == "unlocatable", "excluded")
      .otherwise("eligible"))

ff2 = (ff
    .join(mother.select("unique_id", "mother_conditions_effective", "mother_standin_p"),
          "unique_id", "left")
    .withColumn("mother_conditions_effective", F.coalesce(F.col("mother_conditions_effective"), F.lit(0.0)))
    .withColumn("mother_standin_p", F.coalesce(F.col("mother_standin_p"), F.lit(0.0)))
    .withColumn("model_eligibility", me))

ff2.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.care_finder.facilities_final")

v = spark.table("workspace.care_finder.facilities_final")
print("rows:", v.count())
display(v.groupBy("model_eligibility").count())
cs = (v.join(spark.table("workspace.care_finder.facility_service_summary")
            .filter("service = 'c_section' and n_citations > 0"), "unique_id"))
print("c_section candidates in J (eligible):", cs.filter("model_eligibility = 'eligible'").count())
display(v.groupBy(F.when(F.col("mother_conditions_effective") > 0, "has mother evidence")
                  .otherwise("none"))
         .agg(F.count("*").alias("facilities"), F.round(F.avg("mother_standin_p"), 3).alias("avg_p")))

In [ ]:
# ---- Step 15: district coverage + care gaps, joined to NFHS-5 need measures ----
ff = spark.table("workspace.care_finder.facilities_final")
m = spark.table("workspace.care_finder.facility_condition_matrix")
nfhs = spark.table("databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset."
                   "nfhs_5_district_health_indicators")

cov = (m.join(ff.select("unique_id", "pin_state", "pin_district", "model_eligibility"), "unique_id")
    .filter("model_eligibility = 'eligible' and pin_district is not null and pin_state is not null")
    .groupBy("pin_state", "pin_district", "service")
    .agg(F.count("*").alias("n_facilities"),
         F.sum((F.col("best_tier") == "strong").cast("int")).alias("n_strong"),
         F.sum((F.col("best_tier") == "medium").cast("int")).alias("n_medium"),
         F.sum((F.col("best_tier") == "weak").cast("int")).alias("n_weak")))

nf = (nfhs.select(
        F.lower(F.trim(F.col("state_ut"))).alias("d_state"),
        F.lower(F.trim(F.col("district_name"))).alias("d_district"),
        F.col("institutional_birth_5y_pct").alias("inst_birth_pct"),
        F.col("births_delivered_by_csection_5y_pct").alias("csection_birth_pct"),
        F.col("mothers_who_had_at_least_4_anc_visits_lb5y_pct").alias("anc4_pct")))

cov2 = (cov
    .withColumn("d_state", F.lower(F.trim(F.col("pin_state"))))
    .withColumn("d_district", F.lower(F.trim(F.col("pin_district"))))
    .join(nf, ["d_state", "d_district"], "left")
    .withColumnRenamed("d_state", "state").withColumnRenamed("d_district", "district")
    .drop("pin_state", "pin_district"))
cov2.write.mode("overwrite").saveAsTable("workspace.care_finder.district_service_coverage")

conds = [r.service for r in m.select("service").distinct().collect()]
gaps = (nf.select("d_state", "d_district").distinct()
    .crossJoin(spark.createDataFrame([(c,) for c in conds], "gap_service string"))
    .join(cov2, (F.col("d_state") == F.col("state")) & (F.col("d_district") == F.col("district"))
                  & (F.col("gap_service") == F.col("service")), "left")
    .filter("n_facilities is null")
    .select("d_state", "d_district", "gap_service"))
gaps.write.mode("overwrite").saveAsTable("workspace.care_finder.district_care_gaps")

c = spark.table("workspace.care_finder.district_service_coverage")
g = spark.table("workspace.care_finder.district_care_gaps")
print("coverage rows:", c.count(), "| districts:", c.select("state", "district").distinct().count(),
      "| NFHS-matched:", c.filter("inst_birth_pct is not null").count())
print("gap rows:", g.count())
display(g.groupBy("gap_service").count().orderBy(F.desc("count")))

# Steps 16-18: Near-dup merges, 22-service taxonomy with re-attribution, operator class + continuity

**Step 16 - Merge near-duplicate pairs.** The 10 `near_dup_flag` rows are 5 same-name+same-city pairs (Om Dental Clinic, Siri Dental Hospital, Sarla Hospital, GMC Thiruvananthapuram, Care Dental Hospital); every pair has 3-7 conflicting contact fields, so "drop one if identical" never applies. Survivor = more complete row; arrays (`capability_list`, `procedure_list`, `equipment_list`, `source_url_list`, `specialties_tags`) and phone/website tokens are unioned, scalar fields coalesced, absorbed row dropped. `facilities_final` 9,989 -> 9,984; new `near_dup_merged` / `merged_from` columns; audit table `facility_merges`. Verified: 553 districts before and after (no district loses coverage).

**Step 17 - 22-service taxonomy + re-attribution.** Added ventilation, phototherapy, incubator_warmer, pediatric_critical_care, emergency_medicine, maternal_fetal_medicine, ambulance, epidural_painless, sonography_ultrasound, lab_pathology. Key finding: all 554 distinct names cited in other records' text already exist in the dataset, so other-facility claims are **re-attributed to the cited facility** (`is_reattributed`, `mentioned_by`) instead of rejected - fixing Raj-Eye-style false positives at the source. `facility_service_evidence`: 85,878 rows (85,746 accepted, 132 negated, 267 re-attributed of which 264 accepted); `facility_service_summary`: 46,037 rows, `n_other_facility` replaced by `n_reattributed`. Facility counts by service: institutional_delivery 5,226, lab_pathology 3,446, sonography_ultrasound 2,047, ambulance 1,184, ventilation 496, phototherapy 133, incubator_warmer 137, pediatric_critical_care 309, epidural_painless 149.

**Step 18 - Operator class, continuity, rebuilt model inputs.** `facility_condition_matrix` rebuilt from the new summary (16,525 rows, now with `n_reattributed`). `facilities_final` gains `operator_class` (private 8,832 / public 466 / unknown 686) and `continuity_services` (count of antenatal + institutional delivery + newborn care + immunization; 269 facilities cover all 4). Mother stand-in and maternal aggregates recomputed (6,122 facilities with mother evidence; 5,827 with maternal services). `district_service_coverage`: 1,808 rows, 468 districts, 1,299 NFHS-matched; `district_care_gaps`: 2,231 rows. C-section eligible J = 4,246.

In [ ]:
from pyspark.sql import functions as F, Window

# ---------------- Step 16: merge the near-duplicate pairs ----------------
ff = spark.table("workspace.care_finder.facilities_final")
districts_before = ff.filter("pin_district is not null").select("pin_state","pin_district").distinct().count()

def norm(c):
    return F.regexp_replace(F.lower(F.trim(c)), r"\s+", " ")

SCALAR_COLS = ["yearEstablished","numberDoctors","capacity","officialPhone","address_line1",
               "address_line2","address_line3","address_zipOrPostcode","description","email",
               "officialWebsite","facebookLink","organization_type"]
TOKEN_COLS = ["phone_numbers","websites"]
ARRAY_COLS = ["capability_list","procedure_list","equipment_list","source_url_list","specialties_tags"]
completeness = sum(F.col(c).isNotNull().cast("int") for c in SCALAR_COLS)

g = (ff.withColumn("nd_name", norm("name"))
       .withColumn("nd_city", F.coalesce(norm("address_city"), F.lit("")))
       .withColumn("filled", completeness)
       .withColumn("_rn", F.row_number().over(
           Window.partitionBy("nd_name","nd_city")
                 .orderBy(F.desc("filled"), F.desc("n_source_urls"), F.col("unique_id").desc())))
       .withColumn("_cnt", F.count(F.lit(1)).over(Window.partitionBy("nd_name","nd_city"))))

surv = g.filter("_rn = 1 and _cnt > 1").drop("filled","_rn","_cnt")
abso = g.filter("_rn > 1").select(
    "nd_name","nd_city", F.col("unique_id").alias("absorbed_id"),
    *[F.col(c).alias("a_"+c) for c in SCALAR_COLS+TOKEN_COLS+ARRAY_COLS])

j = surv.join(abso, ["nd_name","nd_city"])
for c in ARRAY_COLS:
    j = j.withColumn(c, F.array_distinct(F.concat(
        F.coalesce(F.col(c), F.array()), F.coalesce(F.col("a_"+c), F.array()))))

def union_tokens(a, b):
    parts = F.flatten(F.array(F.split(F.coalesce(F.col(a), F.lit("")), r"\s*[,;]\s*"),
                              F.split(F.coalesce(F.col(b), F.lit("")), r"\s*[,;]\s*")))
    parts = F.filter(parts, lambda x: F.length(F.trim(x)) > 0)
    return (F.when(F.col(a).isNull() & F.col(b).isNull(), F.lit(None).cast("string"))
             .otherwise(F.array_join(F.array_distinct(parts), ", ")))
for c in TOKEN_COLS:
    j = j.withColumn(c, union_tokens(c, "a_"+c))
for c in SCALAR_COLS:
    j = j.withColumn(c, F.coalesce(F.col(c), F.col("a_"+c)))

merged_rows = (j.drop("nd_name","nd_city", *["a_"+c for c in SCALAR_COLS+TOKEN_COLS+ARRAY_COLS])
    .withColumn("near_dup_merged", F.lit(True))
    .withColumn("merged_from", F.col("absorbed_id"))
    .drop("absorbed_id"))

merge_audit = j.select(F.col("unique_id").alias("survivor_id"), F.col("absorbed_id"),
                       F.col("name"), F.col("address_city"),
                       F.to_json(F.struct(*[F.col("a_"+c) for c in SCALAR_COLS+TOKEN_COLS+ARRAY_COLS])).alias("absorbed_data"))
merge_audit.write.mode("overwrite").saveAsTable("workspace.care_finder.facility_merges")

non_pair = (g.filter("_cnt = 1").drop("nd_name","nd_city","filled","_rn","_cnt")
    .withColumn("near_dup_merged", F.lit(False))
    .withColumn("merged_from", F.lit(None).cast("string")))

non_pair.unionByName(merged_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.care_finder.facilities_final")

v = spark.table("workspace.care_finder.facilities_final")
print("rows:", v.count(), "| merged survivors:", v.filter("near_dup_merged").count())
districts_after = v.filter("pin_district is not null").select("pin_state","pin_district").distinct().count()
print("districts before:", districts_before, "| after:", districts_after)

In [ ]:
from pyspark.sql import functions as F
from functools import reduce

# ---------------- Step 17: 22-service taxonomy + evidence rebuild with re-attribution ----------------
ff = spark.table("workspace.care_finder.facilities_final")   # 9,984 rows post-merge

TAXONOMY = [
 ("antenatal_care",       r"(?i)\banc\b|antenatal|prenatal|pregnan", "obstetric|gynec|midwi"),
 ("institutional_delivery", r"(?i)institutional deliver|labour room|labor room|\bdeliver", "obstetric|gynec"),
 ("normal_delivery",      r"(?i)normal deliver|vaginal deliver", "obstetric|gynec"),
 ("c_section",            r"(?i)c[- ]?section|cesar|caesar", "obstetric|gynec"),
 ("emergency_obstetric",  r"(?i)emergency obstetric|obstetric emergen|emoc", "obstetric|gynec"),
 ("newborn_care",         r"(?i)newborn|neonat|\bnicu\b|\bsncu\b|special newborn", "neonat"),
 ("postnatal_care",       r"(?i)postnatal|\bpnc\b|after delivery|post[- ]delivery", "obstetric|gynec"),
 ("family_planning",      r"(?i)family planning|contracept|sterili[sz]ation|tubectomy|vasectomy|\biud\b|intrauterine|oral pill", None),
 ("immunization",         r"(?i)immuni[sz]|vaccin|\bbcg\b|\bpolio\b|measles|pentavalent", None),
 ("breastfeeding_support", r"(?i)breastfeed|lactat|colostrum", None),
 ("blood_bank",           r"(?i)blood bank|blood storage|safe blood|blood transfusion|blood donation|blood unit|storage of blood", None),
 ("emergency_24h",        r"(?i)24[- ]?hours?|24/7|round[- ]the[- ]clock|emergency (care|services|department|room)|casualty", None),
 ("ventilation",          r"(?i)ventilat|\bcpap\b", None),
 ("phototherapy",         r"(?i)phototherapy|bili.?lights?|jaundice (treatment|light|therapy)", None),
 ("incubator_warmer",     r"(?i)incubator|radiant warmer|neonatal warmer", None),
 ("pediatric_critical_care", r"(?i)\bpicu\b|pediatric (intensive|critical)|paediatric (intensive|critical)", None),
 ("emergency_medicine",   r"(?i)emergency medicine|emergency department|casualty", None),
 ("maternal_fetal_medicine", r"(?i)maternal[- ]?fetal|fetal medicine|foetal medicine|high[- ]risk pregnanc", None),
 ("ambulance",            r"(?i)ambulance", None),
 ("epidural_painless",    r"(?i)epidural|painless (delivery|labour|labor)", None),
 ("sonography_ultrasound", r"(?i)sonograph|ultrasound|\busg\b", None),
 ("lab_pathology",        r"(?i)patholog|laborator|blood test|diagnostic (centre|center|lab)", None),
]
tax = spark.createDataFrame(TAXONOMY, "service string, text_regex string, strong_tag_regex string")
tax.write.mode("overwrite").saveAsTable("workspace.care_finder.service_taxonomy")

def norm(c):
    return F.regexp_replace(F.lower(F.trim(c)), r"\s+", " ")

NAME_RE = (r"((?:[A-Z][\w&'.-]*\s+){0,4}[A-Z][\w&'.-]*\s+"
           r"(?:Hospital|Hospitals|Clinic|Nursing Home|Institute|Medical College|"
           r"College|Centre|Center|Maternity Home|Polyclinic|Sanatorium|Trust))")

# sentence universe (post-merge, so absorbed rows' sentences live in the survivor's arrays)
parts = [ff.select("unique_id", "name", F.lit(fld).alias("field"),
                   F.expr(f"posexplode({fld})")).withColumnRenamed("col", "quote")
         for fld in ["capability_list", "procedure_list", "equipment_list"]]
d = (ff.filter("description is not null")
     .select("unique_id", "name", F.split("description", r"(?<=[.!?])\s+").alias("s")))
parts.append(d.select("unique_id", "name", F.lit("description").alias("field"),
                      F.expr("posexplode(s)")).withColumnRenamed("col", "quote"))
all_items = (reduce(lambda a, b: a.unionByName(b), parts)
    .select("unique_id", "name", "field", "pos", "quote")
    .filter(F.length(F.trim("quote")) > 5))

# other-facility citations (rebuilt on merged data; names table carries no unique_id -> no ambiguity)
names_match = (ff.select(norm("name").alias("facility_name")).distinct()
    .filter(F.length("facility_name") >= 8))
hits = (all_items
    .select("unique_id", "name", "field", "pos", "quote",
            F.explode(F.regexp_extract_all(F.col("quote"), F.lit(NAME_RE))).alias("cand"))
    .withColumn("cand_norm", norm(F.col("cand")))
    .join(names_match, F.col("cand_norm") == F.col("facility_name"))
    .filter(F.col("cand_norm") != norm(F.col("name")))
    .groupBy("unique_id", "field", "pos", "quote")
    .agg(F.min("cand_norm").alias("other_facility_name")))
hits.write.mode("overwrite").saveAsTable("workspace.care_finder.facility_other_facility_mentions")

# name -> unique_id map for attribution (min unique_id on collision)
name_map = (ff.select(norm("name").alias("facility_name"), "unique_id")
    .groupBy("facility_name")
    .agg(F.min("unique_id").alias("cited_id"), F.count_distinct("unique_id").alias("n_holders")))

NEG = r"(?i)\b(no|not|none|nor|neither|without|unavailable|planned|referral|referred to)\b"

ev = (all_items.crossJoin(tax)
    .where(F.expr("quote rlike text_regex"))
    .withColumn("tier", F.when(F.col("field") == "procedure_list", "medium").otherwise("weak"))
    .withColumn("is_negated", F.col("quote").rlike(NEG))
    .join(hits, ["unique_id", "field", "pos", "quote"], "left")
    .join(name_map.select(F.col("facility_name").alias("other_facility_name"), "cited_id"),
          "other_facility_name", "left")
    .withColumn("is_reattributed", F.col("other_facility_name").isNotNull())
    .withColumn("attributed_id", F.when(F.col("is_reattributed"), F.col("cited_id"))
                .otherwise(F.col("unique_id")))
    .withColumn("mentioned_by", F.when(F.col("is_reattributed"), F.col("unique_id"))
                .otherwise(F.lit(None).cast("string")))
    .withColumn("accepted", ~F.col("is_negated")))

tag_ev = (ff.select("unique_id", F.expr("posexplode(specialties_tags)"))
    .withColumnRenamed("col", "tag")
    .crossJoin(tax.filter("strong_tag_regex is not null"))
    .where(F.expr("tag rlike strong_tag_regex"))
    .select(F.col("unique_id"), F.col("service"),
            F.lit("strong").alias("tier"), F.lit("specialties").alias("field"),
            F.col("pos"), F.col("tag").alias("quote"),
            F.lit(False).alias("is_negated"), F.lit(False).alias("is_reattributed"),
            F.lit(None).cast("string").alias("mentioned_by"),
            F.lit(None).cast("string").alias("other_facility_name"),
            F.lit(True).alias("accepted")))

evidence = (ev.select(F.col("attributed_id").alias("unique_id"), "service", "tier", "field", "pos",
                      "quote", "is_negated", "is_reattributed", "mentioned_by",
                      "other_facility_name", "accepted")
    .unionByName(tag_ev))
evidence.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.care_finder.facility_service_evidence")

rank = F.when(F.col("tier") == "strong", F.lit(1)).when(F.col("tier") == "medium", F.lit(2)).otherwise(F.lit(3))
(evidence.withColumn("_rank", rank)
    .groupBy("unique_id", "service")
    .agg(F.sum(F.col("accepted").cast("int")).alias("n_citations"),
         F.min(F.when(F.col("accepted"), F.col("_rank"))).alias("_best"),
         F.sum(F.col("is_negated").cast("int")).alias("n_negated"),
         F.sum(F.col("is_reattributed").cast("int")).alias("n_reattributed"))
    .withColumn("best_tier", F.when(F.col("_best") == 1, "strong")
                .when(F.col("_best") == 2, "medium").when(F.col("_best") == 3, "weak"))
    .drop("_best")
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        "workspace.care_finder.facility_service_summary"))

s = spark.table("workspace.care_finder.facility_service_summary").filter("n_citations > 0")
print("evidence rows:", evidence.count(), "| re-attributed:", evidence.filter("is_reattributed").count())
display(s.groupBy("service").agg(F.count("*").alias("facilities")).orderBy(F.desc("facilities")))

In [ ]:
from pyspark.sql import functions as F

# ---------------- Step 18: matrix + operator class + continuity + refreshed model inputs ----------------
ff = (spark.table("workspace.care_finder.facilities_final")
    .drop("mother_conditions_effective", "mother_standin_p",
          "maternal_services", "maternal_best_tier"))   # recomputed below
summ = spark.table("workspace.care_finder.facility_service_summary").filter("n_citations > 0")

CONDS = ["c_section", "institutional_delivery", "newborn_care", "blood_bank", "emergency_24h"]
tier_w = (F.when(F.col("best_tier") == "strong", F.lit(1.0))
          .when(F.col("best_tier") == "medium", F.lit(0.5)).otherwise(F.lit(0.25)))

matrix = (summ.filter(F.col("service").isin(CONDS))
    .join(ff.select("unique_id", "contamination_flag"), "unique_id")
    .withColumn("tier_weight", tier_w)
    .withColumn("condition_weight",
        F.round(F.col("tier_weight") * F.when(F.col("contamination_flag"), F.lit(0.5)).otherwise(F.lit(1.0)), 3))
    .select("unique_id", "service", "n_citations", "n_negated", "n_reattributed",
            "best_tier", "tier_weight", "contamination_flag", "condition_weight"))
matrix.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.care_finder.facility_condition_matrix")
m = spark.table("workspace.care_finder.facility_condition_matrix")

mother = (m.filter("service in ('c_section','blood_bank','emergency_24h','newborn_care')")
    .groupBy("unique_id")
    .agg(F.round(F.sum("condition_weight"), 3).alias("mother_conditions_effective")))
mother = mother.withColumn("mother_standin_p", F.round(F.col("mother_conditions_effective") / 5, 3))

MATERNAL10 = ["antenatal_care","institutional_delivery","normal_delivery","c_section",
              "emergency_obstetric","newborn_care","postnatal_care","family_planning",
              "immunization","breastfeeding_support"]
t_rank = (F.when(F.col("best_tier") == "strong", F.lit(1))
           .when(F.col("best_tier") == "medium", F.lit(2))
           .when(F.col("best_tier") == "weak", F.lit(3)))
maternal = (summ.filter(F.col("service").isin(MATERNAL10))
    .withColumn("_r", t_rank)
    .groupBy("unique_id")
    .agg(F.count_distinct("service").alias("_n"), F.min("_r").alias("_b"))
    .withColumn("maternal_services", F.col("_n"))
    .withColumn("maternal_best_tier",
        F.when(F.col("_b") == 1, "strong").when(F.col("_b") == 2, "medium").otherwise("weak"))
    .select("unique_id", "maternal_services", "maternal_best_tier"))

CONTINUITY = ["antenatal_care","institutional_delivery","newborn_care","immunization"]
cont = (summ.filter(F.col("service").isin(CONTINUITY))
    .groupBy("unique_id").agg(F.count_distinct("service").alias("continuity_services")))

operator_class = (F.when(F.lower(F.col("operatorTypeId")) == "private", "private")
    .when(F.lower(F.col("operatorTypeId")).isin("public", "government"), "public")
    .otherwise("unknown"))
me = (F.when(F.col("location_quality").isin("conflict_unresolved", "pin_approx"), "warning")
      .when(F.col("location_quality") == "unlocatable", "excluded")
      .otherwise("eligible"))

(ff
    .join(mother.select("unique_id", "mother_conditions_effective", "mother_standin_p"), "unique_id", "left")
    .join(maternal, "unique_id", "left")
    .join(cont, "unique_id", "left")
    .withColumn("mother_conditions_effective", F.coalesce(F.col("mother_conditions_effective"), F.lit(0.0)))
    .withColumn("mother_standin_p", F.coalesce(F.col("mother_standin_p"), F.lit(0.0)))
    .withColumn("maternal_services", F.coalesce(F.col("maternal_services"), F.lit(0)))
    .withColumn("continuity_services", F.coalesce(F.col("continuity_services"), F.lit(0)))
    .withColumn("operator_class", operator_class)
    .withColumn("model_eligibility", me)
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        "workspace.care_finder.facilities_final"))

# ---------------- district coverage + gaps (from the new matrix) ----------------
nfhs = spark.table("databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset."
                   "nfhs_5_district_health_indicators")
cov = (m.join(ff.select("unique_id", "pin_state", "pin_district", "model_eligibility"), "unique_id")
    .filter("model_eligibility = 'eligible' and pin_district is not null and pin_state is not null")
    .groupBy("pin_state", "pin_district", "service")
    .agg(F.count("*").alias("n_facilities"),
         F.sum((F.col("best_tier") == "strong").cast("int")).alias("n_strong"),
         F.sum((F.col("best_tier") == "medium").cast("int")).alias("n_medium"),
         F.sum((F.col("best_tier") == "weak").cast("int")).alias("n_weak")))
nf = (nfhs.select(
        F.lower(F.trim(F.col("state_ut"))).alias("d_state"),
        F.lower(F.trim(F.col("district_name"))).alias("d_district"),
        F.col("institutional_birth_5y_pct").alias("inst_birth_pct"),
        F.col("births_delivered_by_csection_5y_pct").alias("csection_birth_pct"),
        F.col("mothers_who_had_at_least_4_anc_visits_lb5y_pct").alias("anc4_pct")))
cov2 = (cov
    .withColumn("d_state", F.lower(F.trim(F.col("pin_state"))))
    .withColumn("d_district", F.lower(F.trim(F.col("pin_district"))))
    .join(nf, ["d_state", "d_district"], "left")
    .withColumnRenamed("d_state", "state").withColumnRenamed("d_district", "district")
    .drop("pin_state", "pin_district"))
cov2.write.mode("overwrite").saveAsTable("workspace.care_finder.district_service_coverage")

conds = [r.service for r in m.select("service").distinct().collect()]
(nf.select("d_state", "d_district").distinct()
    .crossJoin(spark.createDataFrame([(c,) for c in conds], "gap_service string"))
    .join(cov2, (F.col("d_state") == F.col("state")) & (F.col("d_district") == F.col("district"))
                  & (F.col("gap_service") == F.col("service")), "left")
    .filter("n_facilities is null")
    .select("d_state", "d_district", "gap_service")
    .write.mode("overwrite").saveAsTable("workspace.care_finder.district_care_gaps"))

v = spark.table("workspace.care_finder.facilities_final")
print("rows:", v.count(), "| matrix rows:", m.count())
display(v.groupBy("operator_class").count())
display(v.groupBy("continuity_services").agg(F.count("*").alias("facilities")).orderBy("continuity_services"))
c = spark.table("workspace.care_finder.district_service_coverage")
g = spark.table("workspace.care_finder.district_care_gaps")
print("coverage rows:", c.count(), "| districts:", c.select("state","district").distinct().count(),
      "| gap rows:", g.count())